In [1]:
import pandas as pd
import duckdb
import numpy as np

In [2]:
# Load cleaned transactions and historical team records.
team_seasons = pd.read_csv("../data/interim/team_season_records.csv")
transactions = pd.read_csv("../data/interim/cleaned_trades_fixed.csv")
all_games = pd.read_csv("../data/interim/all_regular_season_games.csv")

In [3]:
team_seasons.head()

,Unnamed: 0,Season,teamID,Wins,teamName,Losses,Games,season_start_year,season_start_date,season_end_date
0,0,1976-77,1610612737,31,Hawks,51,82,1976,1976-10-21,1977-04-10
1,1,1976-77,1610612738,44,Celtics,38,82,1976,1976-10-21,1977-04-10
2,2,1976-77,1610612739,43,Cavaliers,39,82,1976,1976-10-21,1977-04-10
3,3,1976-77,1610612741,44,Bulls,38,82,1976,1976-10-21,1977-04-10
4,4,1976-77,1610612743,50,Nuggets,32,82,1976,1976-10-21,1977-04-10


In [4]:
transactions.head()

,Unnamed: 0,index,Date,Team,Acquired,Relinquished,Notes,page_url
0,0,0,1976-11-16,Lakers,"1977 first round pick (#15-Brad Davis), cash",Mack Calvin,trade with Spurs,http://prosportstransactions.com/basketball/Se...
1,1,1,1976-11-16,Spurs,Mack Calvin,"1977 first round pick (#15-Brad Davis), cash",trade with Lakers,http://prosportstransactions.com/basketball/Se...
2,2,2,1976-11-18,Bulls,John Mengelt,cash,trade with Pistons,http://prosportstransactions.com/basketball/Se...
3,3,3,1976-11-18,Pistons,cash,John Mengelt,trade with Bulls,http://prosportstransactions.com/basketball/Se...
4,4,4,1976-11-30,Bulls,"1977 second round pick (#23-Mike Glenn), cash",Bob Love,trade with Nets,http://prosportstransactions.com/basketball/Se...


In [5]:
# Normalize all date columns before record reconstruction.
transactions["Date"] = pd.to_datetime(transactions["Date"])
team_seasons["season_start_date"] = pd.to_datetime(team_seasons["season_start_date"])
team_seasons["season_end_date"] = pd.to_datetime(team_seasons["season_end_date"])
all_games["game_Date"] = pd.to_datetime(all_games["game_Date"])

In [6]:
# Create one set of calendar boundaries per season.
season_boundaries = team_seasons[["Season", "season_start_date", "season_end_date"]].drop_duplicates().reset_index(drop=True)

In [7]:
# Reconstruct each team record as of the transaction date.
def reconstruct_record(transaction_date, team):
    transaction_date = pd.Timestamp(transaction_date).normalize()

    empty_stats = {
        "teamID": pd.NA,
        "current_Wins": pd.NA,
        "current_Losses": pd.NA,
        "games_Played": pd.NA,
        "remaining_Wins": pd.NA,
        "remaining_Losses": pd.NA,
    }

    # Step 1: Determine whether the date falls inside a regular season
    date_match = season_boundaries.loc[
        (season_boundaries["season_start_date"] <= transaction_date) & (season_boundaries["season_end_date"] >= transaction_date)
    ]

    if date_match.empty:
        return {"Season": "Offseason", **empty_stats, "date_Match": False, "team_Match": pd.NA, "record_Status": "outside_regular_season"}

    matched_seasons = date_match["Season"].drop_duplicates()

    if len(matched_seasons) > 1:
        raise ValueError(f"Multiple seasons found for date " f"{transaction_date.date()}.")

    season = matched_seasons.iloc[0]
    season_start_date = date_match.iloc[0]["season_start_date"]

    # Step 2: Check whether this exact team name exists in that season
    team_match = team_seasons.loc[(team_seasons["Season"] == season) & (team_seasons["teamName"] == team)].drop_duplicates(
        subset=["Season", "teamID", "teamName"]
    )

    if team_match.empty:
        return {"Season": season, **empty_stats, "date_Match": True, "team_Match": False, "record_Status": "team_name_not_found_in_season"}

    if len(team_match) > 1:
        raise ValueError(f"Multiple team matches found for team {team} " f"in season {season}.")

    season_row = team_match.iloc[0]
    team_id = season_row["teamID"]

    # Step 3: Calculate the team's record before the transaction
    games_to_date = all_games.loc[
        (all_games["Season"] == season)
        & (all_games["game_Date"] >= season_start_date)
        & (all_games["game_Date"] < transaction_date)
        & ((all_games["hometeamId"] == team_id) | (all_games["awayteamId"] == team_id))
    ]

    wins = (games_to_date["winner"] == team_id).sum()

    games_played = len(games_to_date)
    losses = games_played - wins

    return {
        "Season": season,
        "teamID": team_id,
        "current_Wins": int(wins),
        "current_Losses": int(losses),
        "games_Played": int(games_played),
        "remaining_Wins": (int(season_row["Wins"]) - int(wins)),
        "remaining_Losses": (int(season_row["Losses"]) - int(losses)),
        "date_Match": True,
        "team_Match": True,
        "record_Status": "matched",
    }

In [8]:
# Normalize historical team aliases and attach pre/post records.
transactions = transactions.copy()

team_name_map = {"Blazers": "Trail Blazers", "Sonics": "SuperSonics"}

transactions["Team"] = transactions["Team"].replace(team_name_map)

record_columns = ["teamID", "current_Wins", "current_Losses", "games_Played", "remaining_Wins", "remaining_Losses"]

# Ensure the new columns exist
for column in ["Season"] + record_columns:
    if column not in transactions.columns:
        transactions[column] = pd.NA

for index, row in transactions.iterrows():
    try:
        record = reconstruct_record(row["Date"], row["Team"])

        for column, value in record.items():
            transactions.loc[index, column] = value

    except ValueError as error:
        # Only treat "no active season" as an offseason transaction
        if "No active regular season found" in str(error):
            transactions.loc[index, "Season"] = "Offseason"
            transactions.loc[index, record_columns] = pd.NA
        else:
            # Preserve other errors, such as duplicate season matches
            raise

In [9]:
transactions["Season"].value_counts()

Season
Offseason    2374
2018-19        61
2014-15        57
2008-09        53
2004-05        44
1979-80        42
2009-10        42
2010-11        41
1977-78        41
1982-83        39
2003-04        38
2016-17        38
2013-14        38
2017-18        38
1987-88        32
2005-06        32
2012-13        31
1995-96        30
1976-77        30
1981-82        30
2007-08        30
2015-16        30
1989-90        28
1996-97        28
1997-98        28
1978-79        28
1980-81        24
1986-87        24
2011-12        24
1993-94        22
2000-01        20
2006-07        18
1990-91        17
1988-89        16
1992-93        16
1983-84        14
1985-86        14
1984-85        14
1994-95        14
1998-99        13
1991-92        12
2001-02        11
2002-03        11
1999-00        10
Name: count, dtype: int64

In [10]:
transactions["record_Status"].value_counts()

record_Status
outside_regular_season    2374
matched                   1223
Name: count, dtype: int64

In [11]:
# Save transactions with point-in-time win-loss context.
transactions.to_csv("../data/interim/transactions_with_WL.csv")